In [ ]:
from pathlib import Path

import pandas as pd

!noglob scp -r hkqai:~/workdir/cc2cc_test5/validate/*.csv ~/workspace/2025.1/validate
name_mol_list = [
    "molecule_W4_11",
    "molecule_G21EA",
    "molecule_G21IP",
    "molecule_DIPCS10",
    "molecule_PA26",
    "molecule_SIE4x4",
    "molecule_ALKBDE10",
    "molecule_YBDE18",
    "molecule_AL2X6",
    "molecule_HEAVYSB11",
    "molecule_NBPRC",
    "molecule_ALK8",
    "molecule_RC21",
    "molecule_G2RC",
    "molecule_BH76",
    "molecule_FH51",
    "molecule_TAUT15",
    "molecule_DC13",
    "molecule_MB16_43",
    "molecule_DARC",
    "molecule_RSE43",
    "molecule_BSR36",
    "molecule_CDIE20",
    "molecule_ISO34",
    "molecule_ISOL24",
    "molecule_C60ISO",
    "molecule_PArel",
    "molecule_BHPERI",
    "molecule_BHDIV10",
    "molecule_INV24",
    "molecule_BHROT27",
    "molecule_PX13",
    "molecule_WCPT18",
    "molecule_RG18",
    "molecule_ADIM6",
    "molecule_S22",
    "molecule_S66",
    "molecule_HEAVY28",
    "molecule_WATER27",
    "molecule_CARBHB12",
    "molecule_PNICO23",
    "molecule_HAL59",
    "molecule_AHB21",
    "molecule_CHB6",
    "molecule_IL16",
    "molecule_IDISP",
    "molecule_ICONF",
    "molecule_ACONF",
    "molecule_Amino20x4",
    "molecule_PCONF21",
    "molecule_MCONF",
    "molecule_SCONF",
    "molecule_UPU23",
    "molecule_BUT14DIOL",
]

if_start_file = {}

for i, name_mol in enumerate(name_mol_list):
    data_path_list = sorted(
        list(Path("../validate").glob(f"*{name_mol}.csv")),
        key=lambda p: p.stat().st_ctime,
    )
    
    for data_path in data_path_list:
        summary_data_path = data_path.stem.split("_" + name_mol)[0] + ".csv"

        with open(data_path, "r") as f:
            data = pd.read_csv(f)

        if if_start_file.get(summary_data_path, True):
            with open(Path("../validate") / summary_data_path, "w") as f2:
                data.to_csv(f2, index=False)
            if_start_file[summary_data_path] = False
        else:
            with open(Path("../validate") / summary_data_path, "a") as f2:
                data.to_csv(f2, index=False, header=False)

for i, name_mol in enumerate(name_mol_list):
    for data_path in list(Path("../validate").glob(f"*{name_mol}.csv")):
        data_path.unlink()

for data_path in list(Path("../validate").glob(f"*test*.csv")):
    data_path.unlink()

scp: /home/chenzihao/workdir/cc2cc_test5/validate/*.csv: No such file or directory


In [ ]:
import json
from pathlib import Path
from datetime import datetime

import pandas as pd
import numpy as np

date = datetime.now().strftime("%Y-%m-%d")

verbose = 2
data_set = "filtered_gmtkn-cc-pVDZ"

data_path_list = sorted(
    list(Path("../validate").glob(f"*gmtkn*.csv")),
    key=lambda p: p.stat().st_ctime,
)

basis_args = "cc-pVDZ"
print(basis_args)

with open(f"../cc2cc/utils/{data_set}.json") as f:
    json_data = json.load(f)

full_subset_dict = {
    "test": ["AHB21"],
    # "sub1": [
    #     "W4_11",
    #     "G21EA",
    #     "G21IP",
    #     "DIPCS10",
    #     "PA26",
    #     "SIE4x4",
    #     "ALKBDE10",
    #     "YBDE18",
    #     "AL2X6",
    #     "HEAVYSB11",
    #     "NBPRC",
    #     "ALK8",
    #     "RC21",
    #     "G2RC",
    #     "BH76RC",
    #     "FH51",
    #     "TAUT15",
    #     "DC13",
    # ],
    # "sub2": [
    #     "MB16_43",
    #     "DARC",
    #     "RSE43",
    #     "BSR36",
    #     "CDIE20",
    #     "ISO34",
    #     "ISOL24",
    #     "C60ISO",
    #     "PArel",
    # ],
    # "sub3": [
    #     "BH76",
    #     "BHPERI",
    #     "BHDIV10",
    #     "INV24",
    #     "BHROT27",
    #     "PX13",
    #     "WCPT18",
    # ],
    # "sub4": [
    #     "RG18",
    #     "ADIM6",
    #     "S22",
    #     "S66",
    #     "HEAVY28",
    #     "WATER27",
    #     "CARBHB12",
    #     "PNICO23",
    #     "HAL59",
    #     "AHB21",
    #     "CHB6",
    #     "IL16",
    # ],
    # "sub5": [
    #     "IDISP",
    #     "ICONF",
    #     "ACONF",
    #     "Amino20x4",
    #     "PCONF21",
    #     "MCONF",
    #     "SCONF",
    #     "UPU23",
    #     "BUT14DIOL",
    # ],
}
subset_list = []
for i_subset in full_subset_dict.keys():
    subset_list.extend([f"{i_subset}_{ i_set}" for i_set in full_subset_dict[i_subset]])

# accumulate summary dictionaries for each file
summary_list = []
subset_summary_list = []

for name_set, subset_list_ in full_subset_dict.items():
    summary_data_list = {}
    for i_subset in subset_list_:
        summary = {}

        for data_path in data_path_list:
            data = pd.read_csv(data_path)
            data["name"] = data["name"].str.split(f"_{basis_args}").str[0]

            data_name = []
            data_reaction_energy_dft = []
            data_reaction_energy_ai = []

            reaction_dict = json_data[f"reaction-{i_subset}"]
            reaction_dict_copy = reaction_dict.copy()
            for i_reaction_name, i_reaction in reaction_dict_copy.items():
                systems_list = i_reaction["systems"]
                stoichiometry_list = i_reaction["stoichiometry"]

                atomic_energy_dft = 0
                atomic_energy_ai = 0
                for i in range(len(systems_list)):
                    finished = True
                    mole_name = (
                        f"{i_subset}-{systems_list[i]}"
                        if i_subset != "BH76RC"
                        else systems_list[i]
                    )

                    if mole_name in json_data:
                        if isinstance(json_data[mole_name], str):
                            mole_name = json_data[mole_name]
                    else:
                        finished = False
                        reaction_dict.pop(i_reaction_name)
                        break

                    col = data["name"] == mole_name
                    if col.any():
                        atomic_energy_dft += data[col]["error_dft_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                        atomic_energy_ai += data[col]["error_scf_ene"].values[0] * int(
                            stoichiometry_list[i]
                        )
                    else:
                        finished = False
                        if verbose > 0:
                            print(
                                f"Warning: {mole_name} not found in {data_path.stem} data file"
                            )
                        break

                if finished:
                    data_reaction_energy_dft.append(atomic_energy_dft)
                    data_reaction_energy_ai.append(atomic_energy_ai)
                    data_name.append(i_reaction_name)

            data_name = np.array(data_name)
            data_reaction_energy_dft = np.array(data_reaction_energy_dft)
            data_reaction_energy_ai = np.array(data_reaction_energy_ai)

            if verbose > 0:
                dft_error_argsort = np.argsort(np.abs(data_reaction_energy_dft))[::-1]
                ai_error_argsort = np.argsort(np.abs(data_reaction_energy_ai))[::-1]

                print(f"####data_path:{data_path}####")
                
                # print(f"====DFT error of {i_subset}====")
                # print(
                #     [
                #         json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                #         for i in dft_error_argsort
                #     ]
                # )
                # print(data_reaction_energy_dft[dft_error_argsort])
                # if verbose == 2:
                #     print(f"====Detail of DFT error of {i_subset}====")
                #     for i in dft_error_argsort:
                #         print(f"{data_name[i]}: {data_reaction_energy_dft[i]}")
                #         systems_list = json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                #         stoichiometry_list = json_data[f"reaction-{i_subset}"][data_name[i]]["stoichiometry"]
                #         for j in range(len(systems_list)):
                #             mole_name = (
                #                 f"{i_subset}-{systems_list[j]}"
                #                 if i_subset != "BH76RC"
                #                 else systems_list[j]
                #             )

                #             if mole_name in json_data:
                #                 if isinstance(json_data[mole_name], str):
                #                     mole_name = json_data[mole_name]

                #             col = data["name"] == mole_name
                #             print(
                #                 data[col]["error_dft_ene"].values[0],
                #                 int(stoichiometry_list[j]),
                #                 systems_list[j],
                #             )
                #         print()
                            
                print(f"====AI error of {i_subset}====")
                print(
                    [
                        json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        for i in ai_error_argsort
                    ]
                )
                print(np.array2string(data_reaction_energy_ai[ai_error_argsort], separator=',',))
                if verbose == 2:
                    print(f"====Detail of AI error of {i_subset}====")
                    for i in ai_error_argsort[:5]:
                        print(f"{data_name[i]}: {data_reaction_energy_ai[i]}")
                        systems_list = json_data[f"reaction-{i_subset}"][data_name[i]]["systems"]
                        stoichiometry_list = json_data[f"reaction-{i_subset}"][data_name[i]]["stoichiometry"]
                        for j in range(len(systems_list)):
                            mole_name = (
                                f"{i_subset}-{systems_list[j]}"
                                if i_subset != "BH76RC"
                                else systems_list[j]
                            )

                            if mole_name in json_data:
                                if isinstance(json_data[mole_name], str):
                                    mole_name = json_data[mole_name]

                            col = data["name"] == mole_name
                            print(
                                data[col]["error_scf_ene"].values[0],
                                int(stoichiometry_list[j]),
                                systems_list[j],
                            )
                        print()
                            
            summary.update(
                {
                    f"{data_path.stem} AI AE": (
                        np.mean(np.abs(data_reaction_energy_ai))
                        if len(data_reaction_energy_ai)
                        else 0
                    ),
                    f"{data_path.stem} DFT AE": (
                        np.mean(np.abs(data_reaction_energy_dft))
                        if len(data_reaction_energy_dft)
                        else 0
                    ),
                    f"{data_path.stem} Processed": f"{len(data_reaction_energy_dft)} / {len(reaction_dict)}",
                }
            )

            if f"{data_path.stem} AI AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} AI AE"] = data_reaction_energy_ai
            else:
                summary_data_list[f"{data_path.stem} AI AE"] = np.append(
                    summary_data_list[f"{data_path.stem} AI AE"],
                    data_reaction_energy_ai,
                )

            if f"{data_path.stem} DFT AE" not in summary_data_list:
                summary_data_list[f"{data_path.stem} DFT AE"] = data_reaction_energy_dft
            else:
                summary_data_list[f"{data_path.stem} DFT AE"] = np.append(
                    summary_data_list[f"{data_path.stem} DFT AE"],
                    data_reaction_energy_dft,
                )

            if f"{data_path.stem} reaction_dict" not in summary_data_list:
                summary_data_list[f"{data_path.stem} reaction_dict"] = len(
                    reaction_dict
                )
            else:
                summary_data_list[f"{data_path.stem} reaction_dict"] += len(
                    reaction_dict
                )

        summary_list.append(summary)

    subset_summary = {}
    for data_path in data_path_list:
        subset_summary.update(
            {
                f"{data_path.stem} AI AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} AI AE"]))
                    if len(summary_data_list[f"{data_path.stem} AI AE"])
                    else 0
                ),
                f"{data_path.stem} DFT AE": (
                    np.mean(np.abs(summary_data_list[f"{data_path.stem} DFT AE"]))
                    if len(summary_data_list[f"{data_path.stem} DFT AE"])
                    else 0
                ),
                f"{data_path.stem} Processed": f"{len(summary_data_list[f"{data_path.stem} AI AE"])} / {summary_data_list[f"{data_path.stem} reaction_dict"]}",
            }
        )
    subset_summary_list.append(subset_summary)

# display one summary table for all files
header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    subset_summary_list,
    index=full_subset_dict.keys(),
)
df_summary.columns = header
display(df_summary)

# save summary to csv with date
df_summary.to_csv(f"../validate/summary_set_{date}.csv")

header = pd.MultiIndex.from_product(
    [
        [data_path.stem for data_path in data_path_list],
        ["AI AE", "DFT AE", "Processed"],
    ],
    names=["data_path", "Type"],
)
df_summary = pd.DataFrame(
    summary_list,
    index=subset_list,
)
df_summary.columns = header

# df_summary.sort_values(
#     by=("ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn-cc-pVDZ", "AI AE"),
#     axis=0,
#     inplace=True,
#     ascending=False,
# )

display(df_summary)

# save summary to csv with date
df_summary.to_csv(f"../validate/summary_subset_{date}.csv")

cc-pVDZ
####data_path:../validate/ccdft_cc-pVDZ_atom-1-543942_d3bj_gmtkn.csv####
====AI error of AHB21====
[['18', '18A', '18B'], ['11', '11A', '11B'], ['14', '14A', '14B'], ['10', '10A', '10B'], ['12', '12A', '12B'], ['19', '19A', '19B'], ['4', '4A', '4B'], ['15', '15A', '15B'], ['8', '8A', '8B'], ['5', '5A', '5B'], ['2', '2A', '2B'], ['13', '13A', '13B'], ['16', '16A', '16B'], ['7', '7A', '7B'], ['9', '9A', '9B'], ['6', '6A', '6B'], ['20', '20A', '20B'], ['1', '1A', '1B'], ['3', '3A', '3B'], ['21', '21A', '21B'], ['17', '17A', '17B']]
[-2.98600656,-2.4174828 ,-2.13779745,-2.11809455,-2.08859648,-2.00740639,
 -1.79849272,-1.61136158, 1.2833471 ,-1.16566237,-0.97429472,-0.87649274,
 -0.81072256,-0.73286398, 0.51681238,-0.42279292,-0.27837244, 0.25390478,
  0.23602039, 0.17245951,-0.11156048]
====Detail of AI error of AHB21====
17: -2.986006563555338
6.396564003603583 1 18
6.832882498830071 -1 18A
2.54968806832885 -1 18B

10: -2.4174827998508848
9.85044552751116 1 11
11.279453241617333 

data_path ccdft_cc-pVDZ_atom-1-543942_d3bj_gmtkn                      \
Type                                       AI AE    DFT AE Processed   
test                                    1.190502  3.464385   21 / 21   

data_path ccdft_cc-pVDZ_atom-1-1150169_gmtkn                      \
Type                                   AI AE    DFT AE Processed   
test                                1.766602  2.453821   21 / 21   

data_path ccdft_cc-pVDZ_atom-1-543942_d3bj_filtered_gmtkn                      \
Type                                                AI AE    DFT AE Processed   
test                                             1.190502  3.464385   21 / 21   

data_path ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn                      
Type                                        AI AE    DFT AE Processed  
test                                     3.116596  2.453821   21 / 21

data_path  ccdft_cc-pVDZ_atom-1-543942_d3bj_gmtkn                      \
Type                                        AI AE    DFT AE Processed   
test_AHB21                               1.190502  3.464385   21 / 21   

data_path  ccdft_cc-pVDZ_atom-1-1150169_gmtkn                      \
Type                                    AI AE    DFT AE Processed   
test_AHB21                           1.766602  2.453821   21 / 21   

data_path  ccdft_cc-pVDZ_atom-1-543942_d3bj_filtered_gmtkn            \
Type                                                 AI AE    DFT AE   
test_AHB21                                        1.190502  3.464385   

data_path            ccdft_cc-pVDZ_atom-1-3881456_4750_gmtkn            \
Type       Processed                                   AI AE    DFT AE   
test_AHB21   21 / 21                                3.116596  2.453821   

data_path             
Type       Processed  
test_AHB21   21 / 21